In [28]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Input
from tensorflow.keras.optimizers import Adam

In [ ]:
df = pd.read_csv('data/data.csv', parse_dates=['Date'], index_col='Date')
print(df.head())

            Temperature
Date                   
2010-01-01    27.483571
2010-01-02    24.308678
2010-01-03    28.238443
2010-01-04    32.615149
2010-01-05    23.829233


In [30]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)

In [31]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


time_step = 100
X, y = create_dataset(scaled_data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [33]:
model = Sequential([
    Input(shape = (X_train.shape[1], 1)),
    GRU(units = 64, return_sequences=True),
    GRU(units = 64, return_sequences=False),
    Dense(1),
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 100, 64)        │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,889 (148.00 KB)

 Trainable params: 37,889 (148.00 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

In [35]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data = (X_test, y_test))

Epoch 1/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0232 - val_loss: 0.0180
Epoch 2/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.0182 - val_loss: 0.0183
Epoch 3/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.0178 - val_loss: 0.0182
Epoch 4/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0181 - val_loss: 0.0178
Epoch 5/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - loss: 0.0178 - val_loss: 0.0178
Epoch 6/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - loss: 0.0180 - val_loss: 0.0180
Epoch 7/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0177 - val_loss: 0.0177
Epoch 8/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0178 - val_loss: 0.0177
Epoch 9/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0177 - val_loss: 0.0180
Epoch 10/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0177 - val_loss: 0.0188


In [36]:
input_sequence = scaled_data[-time_step:].reshape(1, time_step, 1)
predicted_values = model.predict(input_sequence)

predicted_values = scaler.inverse_transform(predicted_values)
print(f"The predicted temperature for the next day is: {predicted_values[0][0]:.2f}°C")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step
The predicted temperature for the next day is: 26.14°C


# Next Word Generator

In [37]:
import tensorflow as tf
from keras.preprocessing.sequence import pad_sequences
from keras.layers import Input, GRU, Dense, TextVectorization, Embedding
from keras import Sequential

In [66]:
with open('blue_castle.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

# Clean up any BOM marker and multiple spaces/newlines
text = text.replace('\ufeff', '').replace('\n', ' ')
words = text.split()
print(f'Total words in text: {len(words)}')
print('Sample text snippet:', ' '.join(words[:50]))

Total words in text: 71314
Sample text snippet: the project gutenberg ebook of the blue castle, by lucy maud montgomery this ebook is for the use of anyone anywhere in the united states and most other parts of the world at no cost and with almost no restrictions whatsoever. you may copy it, give it away or re-use


In [67]:
# We will create sequences of words to train the model
# Let's say we want sequences of 10 words to predict the 11th word
sequence_length = 10

sentences = []
next_words = []
for i in range(0, len(words) - sequence_length):
    sentences.append(' '.join(words[i:i + sequence_length]))
    next_words.append(words[i + sequence_length])

print(f'Total sequences: {len(sentences)}')

# Set up Tokenizer
max_tokens = 10000
tokenizer = TextVectorization(
    max_tokens=max_tokens,
    output_sequence_length=sequence_length
)
tokenizer.adapt(sentences)

# Vectorize input sequences
X_text = tokenizer(sentences).numpy()

# Tokenize output words (we need target indices)
output_tokenizer = TextVectorization(max_tokens=max_tokens, output_sequence_length=1)
output_tokenizer.adapt(next_words)
y_text = output_tokenizer(next_words).numpy().squeeze()

print('Input sequences shape:', X_text.shape)
print('Output values shape:', y_text.shape)

Total sequences: 71304
Input sequences shape: (71304, 10)
Output values shape: (71304,)


In [68]:
# Define model architecture
vocabulary_size = max_tokens
embedding_dim = 128

next_word_model = Sequential([
    Input(shape=(sequence_length,)),
    Embedding(input_dim=vocabulary_size, output_dim=embedding_dim),
    GRU(1024, return_sequences=True),
    GRU(1024),
    Dense(vocabulary_size, activation='softmax')
])

next_word_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

next_word_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 10, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_10 (GRU)                    │ (None, 10, 1024)       │     3,545,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_11 (GRU)                    │ (None, 1024)           │     6,297,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10000)          │    10,250,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,372,688 (81.53 MB)

 Trainable params: 21,372,688 (81.53 MB)

 Non-trainable params: 0 (0.00 B)

In [70]:
# Re-fit the next-word prediction model for a few epochs on the clean vocabulary
next_word_model.fit(
    X_text,
    y_text,
    epochs=10,
    batch_size=128
)

Epoch 1/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.3302 - loss: 3.2029
Epoch 2/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 25s 44ms/step - accuracy: 0.3980 - loss: 2.7946
Epoch 3/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.4647 - loss: 2.4313
Epoch 4/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.5266 - loss: 2.1115
Epoch 5/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.5856 - loss: 1.8236
Epoch 6/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.6436 - loss: 1.5627
Epoch 7/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.6987 - loss: 1.3225
Epoch 8/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.7455 - loss: 1.1149
Epoch 9/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.7910 - loss: 0.9236
Epoch 10/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.8357 - loss: 0.7495


In [75]:
def generate_text(seed_text, next_words_count, model, input_vectorizer, output_vectorizer, sequence_length, temperature=0.7):
    """
    Generates subsequent words based on a seed string.

    Parameters:
    - seed_text (str): The starting text phrase (ideally around 10 words).
    - next_words_count (int): How many total new words to generate.
    - model (Sequential): Your trained Keras GRU model.
    - input_vectorizer (TextVectorization): The tokenizer adapted on input sequences (tokenizer).
    - output_vectorizer (TextVectorization): The tokenizer adapted on targets (output_tokenizer).
    - sequence_length (int): Fixed size of your sequence input window (10).
    - temperature (float): Controls creativity. Lower (0.2) is predictable; higher (1.0) is creative.
    """
    output_text = seed_text
    # Extract the vocabulary array from the target tokenizer to map numbers back to words
    target_vocab = output_vectorizer.get_vocabulary()

    for _ in range(next_words_count):
        # 1. Take the last 'sequence_length' words to match what the model expects
        words_list = output_text.lower().split()
        context_words = words_list[-sequence_length:]
        context_string = ' '.join(context_words)

        # 2. Vectorize the context string using your input tokenizer
        # The tokenizer automatically enforces output_sequence_length=10 (padding/truncating)
        vectorized_input = input_vectorizer([context_string]).numpy()

        # 3. Predict the raw probability distribution for the next token
        predicted_probs = model.predict(vectorized_input, verbose=0)[0]

        # 4. Apply Temperature Scaling to prevent repetitive text loops
        # We inject a small epsilon (1e-7) to prevent taking the log of zero
        predicted_probs = np.log(predicted_probs + 1e-7) / temperature
        exp_preds = np.exp(predicted_probs)
        predicted_probs = exp_preds / np.sum(exp_preds)

        # 5. Sample an index based on the adjusted probability distribution spread
        predicted_word_index = np.random.choice(len(predicted_probs), p=predicted_probs)

        # 6. Retrieve the string word directly from the target vocabulary list
        predicted_word = target_vocab[predicted_word_index]

        # Handle padding or unknown tokens gracefully to avoid breaking the text
        if predicted_word in ['', '[UNK]']:
            # Fallback strategy: pick the absolute highest valid prediction if it rolls an UNK
            sorted_indices = np.argsort(predicted_probs)[::-1]
            for idx in sorted_indices:
                if target_vocab[idx] not in ['', '[UNK]']:
                    predicted_word = target_vocab[idx]
                    break

        # Append the successfully translated word to our running sequence string
        output_text += " " + predicted_word

    return output_text

In [76]:
# Test text sequence
seed = "this ebook is for the use of anyone anywhere in"

# Generate 15 words following the seed phrase
generated_story = generate_text(
    seed_text=seed,
    next_words_count=15,
    model=next_word_model,
    input_vectorizer=tokenizer,         # your input vectorizer variable
    output_vectorizer=output_tokenizer, # your target vectorizer variable
    sequence_length=sequence_length,   # 10
    temperature=0.6                     # Balanced creative setting
)

print(generated_story)


this ebook is for the use of anyone anywhere in the united states and most other parts of the world at no cost and with
